# T5 head pruning — part 2: recovery fine-tuning + memory benchmark

This zip already contains your sweep results from part 1 (`results/`), so nothing is re-run.

**Setup:**
1. Upload this `t5-head-pruning.zip` as a **new version** of your `t5_head_pruning` dataset (dataset page → *New Version*), or as a new dataset, and attach it to this notebook. Remove any older version of the project from the notebook's inputs.
2. Settings: **Accelerator → GPU T4 x2**, **Internet → On**.
3. Run cells 1–3 to check, then **Save Version → Save & Run All (Commit)**. Stop the draft session (power button) so it doesn't use GPU quota too. Expected time: about 30–45 min.

In [ ]:
# 1. copy project (with part-1 results) to the writable working dir
import glob, os, shutil
hits = glob.glob('/kaggle/input/**/run_finetune.py', recursive=True)
assert hits, 'run_finetune.py not found: attach the NEW zip as input'
src = os.path.dirname(hits[0])
shutil.copytree(src, '/kaggle/working/t5-head-pruning', dirs_exist_ok=True)
%cd /kaggle/working/t5-head-pruning
assert os.path.exists('results/sweep_translation.json'), 'part-1 results missing'
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# 2. tests (~15 s)
!python tests/test_pruning.py

In [ ]:
# 3. 2-minute check of the fine-tuning loop (saves nothing)
!python run_finetune.py --task translation --quick

In [ ]:
%%bash
# 4. recovery fine-tuning, one task per T4 (logs: log_ft_*.txt)
CUDA_VISIBLE_DEVICES=0 python run_finetune.py --task translation > log_ft_translation.txt 2>&1 &
CUDA_VISIBLE_DEVICES=1 python run_finetune.py --task summarization > log_ft_summarization.txt 2>&1 &
wait
tail -n 8 log_ft_translation.txt log_ft_summarization.txt

In [ ]:
# 5. re-run the efficiency benchmark with the memory fix (one GPU, nothing else running)
!CUDA_VISIBLE_DEVICES=0 python run_benchmark.py --task translation --method gradient --all_levels
!CUDA_VISIBLE_DEVICES=0 python run_benchmark.py --task summarization --method gradient --target 0.4

In [ ]:
# 6. figures + SUMMARY.md, zipped into the Output tab
!python make_plots.py
!cd /kaggle/working && zip -qr results_part2.zip t5-head-pruning/results t5-head-pruning/log_*.txt
from IPython.display import Image, display
for p in sorted(glob.glob('results/finetune_*.png')): display(Image(p))